# 07 — ECOD + Temporal Features
Adds 13 per-symbol temporal features (rolling cancel/trade sums, rolling return std, rolling max |return|, time delta, cancel-to-trade ratio) on top of the 41 existing features, then fits pooled ECOD.

Hypothesis: the 95% → 100% gap is temporal. Market manipulation (spoofing, layering, quote stuffing) plays out over milliseconds; point-in-time features can't see it.

Writes `submission_ecod_temporal.csv`. Requires notebook 01 processed parquet.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import yaml
from src.features import build_temporal_features, feature_names_all
from src.models import score_ecod

cfg = yaml.safe_load((ROOT / 'configs' / 'config.yaml').read_text())
cols = feature_names_all(cfg['features']['raw'])

In [ ]:
proc = ROOT / 'data' / 'processed'
train = build_temporal_features(pd.read_parquet(proc / 'train.parquet'))
test = build_temporal_features(pd.read_parquet(proc / 'test.parquet'))
print(train.shape, test.shape)

In [ ]:
def mat(df):
    X = df[cols].to_numpy(dtype=np.float32, na_value=0.0)
    np.nan_to_num(X, copy=False, nan=0.0, posinf=0.0, neginf=0.0)
    return X

X_tr, X_te = mat(train), mat(test)
scores = score_ecod(X_tr, X_te, cfg['models']['ecod'])
scores.min(), scores.max(), scores.mean()

In [ ]:
out = ROOT / 'outputs' / 'submissions'
pd.DataFrame({'ID': test['index'], 'TARGET': scores}).to_csv(out / 'submission_ecod_temporal.csv', index=False)